# BRONZE LAYER (3 TABLES)

All bronze columns stay **STRING** (raw, as-landed). Typing happens in silver.
`load_time` and `source_file` are stamped inside the `COPY INTO` SELECT so the audit
columns are populated on the same pass rather than by a follow-up UPDATE. update


## B1) — sales_bz (CSV) — Create + Load

In [0]:
from pyspark.sql import functions as F

catalog = "abs_dev"
db_name = "abs_db"


# ============================================================
# 1. CREATE SCHEMA
# ============================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db_name}")
spark.sql(f"USE {catalog}.{db_name}")


# ============================================================
# 2. GET DATA EXTERNAL LOCATION
# ============================================================

base_dir_data = (
    spark.sql("DESCRIBE EXTERNAL LOCATION `abs-external-location`")
    .select("url")
    .collect()[0][0]
    .rstrip("/")
)

landing_zone = f"{base_dir_data}/landing-zone/raw"

print("Base data location:", base_dir_data)
print("Landing zone:", landing_zone)


# ============================================================
# 3. CREATE BRONZE SALES TABLE + DATE LOOKUP (if missing)
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.sales_bz (
    order_id        STRING,
    customer_id     STRING,
    transaction_id  STRING,
    product_id      STRING,
    quantity        STRING,
    discount_amount STRING,
    total_amount    STRING,
    order_date      STRING,
    load_time       TIMESTAMP,
    source_file     STRING,
    `date`          DATE,
    week_part       STRING
)
USING DELTA
""")

# Only needed so the MERGE below doesn't fail if the lookup hasn't been built yet.
# If you populate date_lookup elsewhere, remove this.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.date_lookup (
    `date`    DATE,
    week_part STRING
)
USING DELTA
""")


# ============================================================
# 4. INCREMENTAL INGESTION WITH COPY INTO
# ============================================================

spark.sql(f"""
COPY INTO {catalog}.{db_name}.sales_bz
FROM (
    SELECT
        order_id,
        customer_id,
        transaction_id,
        product_id,
        quantity,
        discount_amount,
        total_amount,
        order_date,
        current_timestamp()          AS load_time,
        _metadata.file_path          AS source_file,
        TO_DATE(order_date, 'M/d/yyyy')  AS `date`,
        CAST(NULL AS STRING)         AS week_part
    FROM '{landing_zone}/sales'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'inferSchema' = 'false'
)
""")


# ============================================================
# 5. ENRICH WEEK_PART FROM DATE LOOKUP
# ============================================================

spark.sql(f"""
MERGE INTO {catalog}.{db_name}.sales_bz AS target
USING (
    SELECT DISTINCT `date`, week_part
    FROM {catalog}.{db_name}.date_lookup
) AS source
ON target.`date` = source.`date`
WHEN MATCHED AND (target.week_part IS NULL OR target.week_part = '') THEN
    UPDATE SET week_part = source.week_part
""")


# ============================================================
# 6. DISPLAY RECENTLY LOADED RECORDS
# ============================================================

display(
    spark.table(f"{catalog}.{db_name}.sales_bz")
         .orderBy(F.col("load_time").desc())
         .limit(20)
)

## B2) — products_bz (JSON CDC) — Create + Load

Change feed: every row carries `operation` (INSERT / UPDATE / DELETE) and `seqNum`,
the monotonically increasing change sequence for that `product_id`.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.products_bz (
  product_id       STRING,
  product_name     STRING,
  product_category STRING,
  product_price    STRING,
  operation        STRING,
  seqNum           STRING,
  load_time        TIMESTAMP,
  source_file      STRING
) USING DELTA
""")

spark.sql(f"""
COPY INTO {catalog}.{db_name}.products_bz
FROM (
  SELECT
    product_id, product_name, product_category, product_price,
    operation, seqNum,
    current_timestamp()  AS load_time,
    _metadata.file_path  AS source_file
  FROM '{base_dir_data}/landing-zone/raw/products'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
""")

display(spark.table(f"{catalog}.{db_name}.products_bz").orderBy(F.col("load_time").desc()).limit(20))

## B3) — customers_bz (JSON CDC) — Create + Load

Same shape as products, but the sequence column is named `sequenceNum`.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{db_name}.customers_bz (
  customer_id     STRING,
  customer_name   STRING,
  customer_email  STRING,
  customer_city   STRING,
  customer_state  STRING,
  operation       STRING,
  sequenceNum     STRING,
  load_time       TIMESTAMP,
  source_file     STRING
) USING DELTA
""")

spark.sql(f"""
COPY INTO {catalog}.{db_name}.customers_bz
FROM (
  SELECT
    customer_id, customer_name, customer_email, customer_city, customer_state,
    operation, sequenceNum,
    current_timestamp()  AS load_time,
    _metadata.file_path  AS source_file
  FROM '{base_dir_data}/landing-zone/raw/customers.csv'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
""")

display(spark.table(f"{catalog}.{db_name}.customers_bz").orderBy(F.col("load_time").desc()).limit(20))